# 01 Data cleaning

> **Simulated case study.** Glowina Skincare is a fictional brand and all data is synthetic.

The raw exports in `data/raw/` contain the kinds of problems that ad platform and order exports usually have. This notebook fixes them and writes analysis-ready tables to `data/clean/`.

Rules I follow throughout:

- Every fix is counted and written to a cleaning log.
- When a value can be recovered from another table, I recover it instead of guessing.
- When it cannot, I impute with a documented rule and flag the row in a `dq_flags` column, so later analysis can exclude or test those rows.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

cleaning_log = []

def log(table, issue, rows, action):
    cleaning_log.append({"table": table, "issue": issue,
                         "rows_affected": int(rows), "action": action})

def flag(df, mask, name):
    """Append a data-quality flag to the rows selected by mask."""
    cur = df["dq_flags"]
    df["dq_flags"] = np.where(mask, np.where(cur == "", name, cur + ";" + name), cur)

def dedupe(df, key):
    """Drop duplicate keys, keeping the copy with the fewest missing values."""
    ranked = df.assign(_nn=df.notna().sum(axis=1)).sort_values("_nn", ascending=False, kind="stable")
    return ranked.drop_duplicates(key, keep="first").drop(columns="_nn").sort_index()

def parse_mixed_date(s):
    """Parse a column that mixes YYYY-MM-DD and DD/MM/YYYY."""
    s = s.astype("string").str.strip()
    has_slash = s.str.contains("/", na=False)
    iso = pd.to_datetime(s.where(~has_slash), format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(s.where(has_slash), format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)

def parse_rupiah(s):
    """'Rp1.250.000' -> 1250000. Plain numbers are kept, a leading minus stays negative."""
    s = s.astype("string").str.strip()
    negative = s.str.startswith("-").fillna(False)
    value = pd.to_numeric(s.str.replace(r"[^0-9]", "", regex=True), errors="coerce")
    return value.where(~negative, -value)

## Load and profile

Everything is read as text first, so nothing is silently converted before I look at it.

In [2]:
names = ["campaigns", "daily_performance", "customers", "orders"]
raw = {n: pd.read_csv(RAW / f"{n}.csv", dtype=str) for n in names}

for n, df in raw.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0].to_dict()
    print(f"{n}: {len(df):,} rows, {df.shape[1]} columns")
    print(f"   missing values: {nulls if nulls else 'none'}")

campaigns: 41 rows, 9 columns
   missing values: {'end_date': 2}
daily_performance: 2,481 rows, 8 columns
   missing values: {'impressions': 27, 'clicks': 42, 'add_to_cart': 25}
customers: 7,103 rows, 7 columns
   missing values: {'city': 281, 'gender': 208, 'age_group': 374}
orders: 12,908 rows, 10 columns
   missing values: {'campaign_id': 3509, 'order_value_idr': 150}

## Campaigns

Problems to expect: channel names written several ways, budgets stored as text such as `Rp 54.000.000`, a repeated campaign and some missing end dates.

In [3]:
camp = raw["campaigns"].apply(lambda s: s.str.strip()).copy()

# 1. Channel labels: reduce to letters only, then map to four canonical names
def label_key(x):
    return re.sub(r"[^a-z]", "", str(x).lower())

CHANNEL_MAP = {
    "metaads": "Meta Ads", "meta": "Meta Ads", "facebookinstagramads": "Meta Ads",
    "tiktokads": "TikTok Ads", "tiktok": "TikTok Ads",
    "googleads": "Google Ads",
    "influencer": "Influencer", "influencermarketing": "Influencer", "kol": "Influencer",
}
keys = camp["channel"].map(label_key)
assert keys.isin(CHANNEL_MAP).all(), keys[~keys.isin(CHANNEL_MAP)].unique()
canonical = keys.map(CHANNEL_MAP)
log("campaigns", "inconsistent channel labels", (camp["channel"] != canonical).sum(), "mapped to 4 canonical names")
print("channel labels before:", sorted(camp["channel"].unique()))
camp["channel"] = canonical
print("channel labels after: ", sorted(camp["channel"].unique()))

# 2. Budgets stored as text
as_text = camp["total_budget_idr"].str.contains("Rp", na=False).sum()
camp["total_budget_idr"] = parse_rupiah(camp["total_budget_idr"]).astype("Int64")
log("campaigns", "budget stored as text ('Rp 54.000.000')", as_text, "parsed to integer IDR")

# 3. Dates
camp["start_date"] = parse_mixed_date(camp["start_date"])
camp["end_date"] = parse_mixed_date(camp["end_date"])

# 4. Repeated campaign
n_dup = camp.duplicated("campaign_id").sum()
camp = dedupe(camp, "campaign_id").reset_index(drop=True)
assert camp["campaign_id"].is_unique
log("campaigns", "duplicate campaign_id", n_dup, "kept one row per campaign")
print(f"\nrows after de-duplication: {len(camp)}  | missing end_date: {camp['end_date'].isna().sum()}")

channel labels before: ['Google Ads', 'GoogleAds', 'Influencer', 'Influencer Marketing', 'KOL', 'Meta Ads', 'TIKTOK ADS', 'TikTok Ads', 'google ads', 'influencer']
channel labels after:  ['Google Ads', 'Influencer', 'Meta Ads', 'TikTok Ads']

rows after de-duplication: 40  | missing end_date: 2

## Daily performance: structure

Dates in two formats, spend stored as text, lower-case campaign IDs and repeated rows. Logic problems (clicks above impressions, negative spend, missing counts) come later, because some of them are repaired with help from the orders table.

In [4]:
d = raw["daily_performance"].apply(lambda s: s.str.strip()).copy()

# Campaign IDs
lower_ids = (d["campaign_id"] != d["campaign_id"].str.upper()).sum()
d["campaign_id"] = d["campaign_id"].str.upper()
log("daily_performance", "lower-case campaign_id", lower_ids, "upper-cased")

# Dates
slash_dates = d["date"].str.contains("/", na=False).sum()
d["date"] = parse_mixed_date(d["date"])
assert d["date"].notna().all()
log("daily_performance", "dates written as DD/MM/YYYY", slash_dates, "converted to ISO dates")

# Spend and counts
spend_text = d["spend_idr"].str.contains("Rp", na=False).sum()
d["spend_idr"] = parse_rupiah(d["spend_idr"])
for c in ["impressions", "clicks", "add_to_cart", "conversions", "revenue_idr"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
log("daily_performance", "spend stored as text ('Rp1.250.000')", spend_text, "parsed to integer IDR")

# Repeated rows: one row per campaign per day
n_dup = d.duplicated(["date", "campaign_id"]).sum()
d = dedupe(d, ["date", "campaign_id"]).reset_index(drop=True)
log("daily_performance", "duplicate campaign-day rows", n_dup, "kept the most complete copy")

assert d["campaign_id"].isin(camp["campaign_id"]).all()
d["dq_flags"] = ""
print(f"rows after de-duplication: {len(d):,}")

# Missing end dates in campaigns: last day the campaign appears in the data
last_seen = d.groupby("campaign_id")["date"].max()
missing_end = camp["end_date"].isna()
camp.loc[missing_end, "end_date"] = camp.loc[missing_end, "campaign_id"].map(last_seen)
log("campaigns", "missing end_date", missing_end.sum(), "filled with the last date seen in daily_performance")
print("campaign end dates still missing:", camp["end_date"].isna().sum())

rows after de-duplication: 2,452
campaign end dates still missing: 0

## Customers

City and gender labels are inconsistent, some fields are empty, and a few customers appear twice. Unknown values are labelled `Unknown` instead of being dropped, so these customers still count in totals.

In [5]:
cu = raw["customers"].apply(lambda s: s.str.strip()).copy()

n_dup = cu.duplicated("customer_id").sum()
cu = dedupe(cu, "customer_id").reset_index(drop=True)
log("customers", "duplicate customer_id", n_dup, "kept the most complete copy")

cu["first_order_date"] = parse_mixed_date(cu["first_order_date"])

CITY_VARIANTS = {
    "Jakarta": ["jakarta", "dki jakarta", "jkt"], "Surabaya": ["surabaya", "sby"],
    "Bandung": ["bandung", "bdg"], "Medan": ["medan"], "Semarang": ["semarang"],
    "Makassar": ["makassar", "mks"], "Yogyakarta": ["yogyakarta", "jogja", "diy"],
    "Denpasar": ["denpasar", "bali"], "Malang": ["malang"], "Sidoarjo": ["sidoarjo"],
    "Tangerang": ["tangerang"], "Bekasi": ["bekasi"], "Depok": ["depok"],
    "Palembang": ["palembang"], "Balikpapan": ["balikpapan"],
}
CITY_MAP = {v: k for k, vs in CITY_VARIANTS.items() for v in vs}
city_key = cu["city"].str.lower()
assert city_key.dropna().isin(CITY_MAP).all(), city_key.dropna()[~city_key.dropna().isin(CITY_MAP)].unique()
city_clean = city_key.map(CITY_MAP)
log("customers", "inconsistent city labels", (cu["city"].notna() & (cu["city"] != city_clean)).sum(), "mapped to 15 canonical cities")
log("customers", "missing city", cu["city"].isna().sum(), "set to 'Unknown'")
cu["city"] = city_clean.fillna("Unknown")

GENDER_MAP = {"f": "Female", "female": "Female", "fem": "Female",
              "m": "Male", "male": "Male", "man": "Male"}
gender_key = cu["gender"].str.lower()
assert gender_key.dropna().isin(GENDER_MAP).all()
gender_clean = gender_key.map(GENDER_MAP)
log("customers", "inconsistent gender labels", (cu["gender"].notna() & (cu["gender"] != gender_clean)).sum(), "mapped to Female / Male")
log("customers", "missing gender", cu["gender"].isna().sum(), "set to 'Unknown'")
cu["gender"] = gender_clean.fillna("Unknown")

log("customers", "missing age_group", cu["age_group"].isna().sum(), "set to 'Unknown'")
cu["age_group"] = cu["age_group"].fillna("Unknown")

print(f"customers after cleaning: {len(cu):,}")
print(cu["city"].value_counts().head(6).to_string())

customers after cleaning: 7,033
city
Jakarta     1500
Surabaya     955
Bandung      690
Medan        511
Semarang     391
Bekasi       367

## Orders

This is the table with the most repairs. The order of the checks matters:

1. Fix labels, dates and signs.
2. Customers that do not exist in the customer table keep their order, because the order still counts toward campaign revenue, but the customer link is cleared and flagged.
3. Missing order values are recovered from the ad platform's daily revenue where only one order is missing on that campaign-day. The platform total minus the known orders gives the missing value exactly. Anything left gets the median order value of the same campaign.

In [6]:
o = raw["orders"].apply(lambda s: s.str.strip()).copy()

n_dup = o.duplicated("order_id").sum()
o = dedupe(o, "order_id").reset_index(drop=True)
log("orders", "duplicate order_id", n_dup, "kept the most complete copy")

slash_dates = o["order_date"].str.contains("/", na=False).sum()
o["order_date"] = parse_mixed_date(o["order_date"])
assert o["order_date"].notna().all()
log("orders", "dates written as DD/MM/YYYY", slash_dates, "converted to ISO dates")

for c in ["items_count", "discount_idr", "order_value_idr"]:
    o[c] = pd.to_numeric(o[c], errors="coerce")
o["dq_flags"] = ""

# Labels
PAYMENT_MAP = {"cod": "COD", "cashondelivery": "COD", "ewallet": "E-Wallet",
               "qris": "QRIS", "va": "Virtual Account", "transferva": "Virtual Account",
               "virtualaccount": "Virtual Account", "cc": "Credit Card", "creditcard": "Credit Card"}
pay_key = o["payment_method"].map(label_key)
assert pay_key.isin(PAYMENT_MAP).all(), pay_key[~pay_key.isin(PAYMENT_MAP)].unique()
pay_clean = pay_key.map(PAYMENT_MAP)
log("orders", "inconsistent payment_method labels", (o["payment_method"] != pay_clean).sum(), "mapped to 5 canonical methods")
o["payment_method"] = pay_clean

STATUS_MAP = {"completed": "Completed", "complete": "Completed", "cancelled": "Cancelled",
              "cancel": "Cancelled", "returned": "Returned", "return": "Returned"}
status_key = o["status"].str.lower()
assert status_key.isin(STATUS_MAP).all()
status_clean = status_key.map(STATUS_MAP)
log("orders", "inconsistent status labels", (o["status"] != status_clean).sum(), "mapped to Completed / Cancelled / Returned")
o["status"] = status_clean

# Negative order values: sign entry errors. Refunds are already modelled by status = Returned.
neg = o["order_value_idr"] < 0
o.loc[neg, "order_value_idr"] = o.loc[neg, "order_value_idr"].abs()
flag(o, neg, "value_sign_fixed")
log("orders", "negative order_value_idr", neg.sum(), "sign flipped (refunds are tracked in status)")

# Customers that are not in the customer table
orphan = ~o["customer_id"].isin(cu["customer_id"])
o.loc[orphan, "customer_id"] = np.nan
flag(o, orphan, "customer_unknown")
log("orders", "customer_id not found in customers", orphan.sum(), "kept the order, cleared customer_id, flagged")

assert o["campaign_id"].dropna().isin(camp["campaign_id"]).all()
print(f"orders after label and sign fixes: {len(o):,}")

orders after label and sign fixes: 12,806

In [7]:
# Missing order values
missing = o["order_value_idr"].isna()
attributed = o["campaign_id"].notna()

# Step A: exact recovery from the platform's daily revenue
by_day = o[attributed].groupby(["campaign_id", "order_date"])["order_value_idr"]
known_sum = by_day.sum()
n_missing = by_day.apply(lambda s: s.isna().sum())
platform_rev = d.set_index(["campaign_id", "date"])["revenue_idr"]
platform_rev.index.names = ["campaign_id", "order_date"]
residual = platform_rev.reindex(known_sum.index) - known_sum
recoverable = residual[(n_missing == 1) & (residual > 0)]

rows = o.index[missing & attributed]
pairs = list(zip(o.loc[rows, "campaign_id"], o.loc[rows, "order_date"]))
recovered = pd.Series([recoverable.get(p, np.nan) for p in pairs], index=rows)
hit = recovered.notna()
o.loc[hit[hit].index, "order_value_idr"] = recovered[hit]
flag(o, o.index.isin(hit[hit].index), "value_recovered_from_platform_revenue")

# Step B: median order value of the same campaign (organic orders use the organic median)
still = o["order_value_idr"].isna()
medians = o.groupby(o["campaign_id"].fillna("ORGANIC"))["order_value_idr"].median()
fill = o["campaign_id"].fillna("ORGANIC").map(medians).round(-3)
o.loc[still, "order_value_idr"] = fill[still]
flag(o, still, "value_imputed_campaign_median")

log("orders", "missing order_value_idr, recovered exactly", hit.sum(), "platform daily revenue minus known orders")
log("orders", "missing order_value_idr, imputed", still.sum(), "median order value of the same campaign")
print(f"missing order values: {missing.sum()}  | recovered exactly: {hit.sum()}  | imputed: {still.sum()}")
assert o["order_value_idr"].notna().all()
o["order_value_idr"] = o["order_value_idr"].astype(int)

missing order values: 147  | recovered exactly: 102  | imputed: 45

### Customer first-order dates

A customer's first order date cannot be later than their earliest order. Where it is, the order table wins.

In [8]:
earliest = o.groupby("customer_id")["order_date"].min()
candidate = cu["customer_id"].map(earliest)
late = candidate.notna() & (candidate < cu["first_order_date"])
cu.loc[late, "first_order_date"] = candidate[late]
log("customers", "first_order_date later than the earliest order", late.sum(), "set to the earliest order date")
print("customers with a corrected first_order_date:", late.sum())

customers with a corrected first_order_date: 71

## Daily performance: logic repairs

Four problems are left: spend below zero, clicks above impressions, conversions above clicks, and missing counts.

- **Negative spend.** Spend cannot be negative for a day an ad ran. I replace it with the campaign's median daily spend and flag the row.
- **Clicks above impressions.** Impressions follow from spend and CPM, which are stable, so the click count is the suspect value. I treat it as missing.
- **Conversions above clicks.** The order table is first-party data, so I replace the platform's conversion count with the number of orders credited to that campaign on that day.
- **Missing counts.** Impressions come from spend and the campaign's median CPM. Clicks come from impressions and the campaign's click-through rate. Add-to-cart comes from the campaign's add-to-cart rate per click.

In [9]:
orders_per_day = (o[o["campaign_id"].notna()]
                  .groupby(["campaign_id", "order_date"]).size()
                  .rename("orders_in_order_table"))
orders_per_day.index.names = ["campaign_id", "date"]
d = d.join(orders_per_day, on=["campaign_id", "date"])
d["orders_in_order_table"] = d["orders_in_order_table"].fillna(0).astype(int)

# 1. Negative spend
bad_spend = d["spend_idr"] < 0
median_spend = d[~bad_spend].groupby("campaign_id")["spend_idr"].median()
d.loc[bad_spend, "spend_idr"] = d.loc[bad_spend, "campaign_id"].map(median_spend).round(-2)
flag(d, bad_spend, "spend_replaced_campaign_median")
log("daily_performance", "negative spend_idr", bad_spend.sum(), "replaced with the campaign's median daily spend")

# 2. Clicks above impressions
bad_clicks = d["clicks"] > d["impressions"]
d.loc[bad_clicks, "clicks"] = np.nan
flag(d, bad_clicks, "clicks_above_impressions")
log("daily_performance", "clicks greater than impressions", bad_clicks.sum(), "clicks set to missing, then imputed")

# 3. Conversions above clicks (clicks already known at this point)
bad_conv = d["conversions"] > d["clicks"]
d.loc[bad_conv, "conversions"] = d.loc[bad_conv, "orders_in_order_table"]
flag(d, bad_conv, "conversions_from_order_table")
log("daily_performance", "conversions greater than clicks", bad_conv.sum(), "replaced with order count from the order table")

In [10]:
# 4. Missing counts
miss_impr = d["impressions"].isna()
miss_click = d["clicks"].isna()
miss_atc = d["add_to_cart"].isna()
print("missing before: impressions", miss_impr.sum(), "| clicks", miss_click.sum(), "| add_to_cart", miss_atc.sum())

# impressions = spend / CPM
cpm = (d["spend_idr"] / d["impressions"] * 1000).groupby(d["campaign_id"]).median()
d.loc[miss_impr, "impressions"] = (d["spend_idr"] / d["campaign_id"].map(cpm) * 1000).round()[miss_impr]

# clicks = impressions x CTR
seen = d[~miss_click & ~miss_impr]
ctr = seen.groupby("campaign_id")["clicks"].sum() / seen.groupby("campaign_id")["impressions"].sum()
d.loc[miss_click, "clicks"] = (d["impressions"] * d["campaign_id"].map(ctr)).round()[miss_click]
d["clicks"] = np.maximum(d["clicks"], d["conversions"])   # clicks cannot be fewer than conversions

# add-to-cart = clicks x campaign add-to-cart rate
ok = ~miss_atc
atc_rate = d[ok].groupby("campaign_id")["add_to_cart"].sum() / d[ok].groupby("campaign_id")["clicks"].sum()
d.loc[miss_atc, "add_to_cart"] = (d["clicks"] * d["campaign_id"].map(atc_rate)).round()[miss_atc]
d["add_to_cart"] = np.maximum(d["add_to_cart"], d["conversions"])

flag(d, miss_impr, "impressions_imputed")
flag(d, miss_click & ~bad_clicks, "clicks_imputed")
flag(d, miss_atc, "add_to_cart_imputed")
log("daily_performance", "missing impressions", miss_impr.sum(), "spend / campaign median CPM")
log("daily_performance", "missing clicks", (miss_click & ~bad_clicks).sum(), "impressions x campaign CTR")
log("daily_performance", "missing add_to_cart", miss_atc.sum(), "clicks x campaign add-to-cart rate")

num_cols = ["impressions", "clicks", "spend_idr", "add_to_cart", "conversions", "revenue_idr"]
d[num_cols] = d[num_cols].astype("int64")
print("missing after:", int(d[num_cols].isna().sum().sum()))

missing before: impressions 27 | clicks 46 | add_to_cart 25
missing after: 0

## Checks after cleaning

The platform and the order table measure the same sales, so they should agree. If they do, the repairs above did not distort the numbers.

In [11]:
first_party = (o[o["campaign_id"].notna()]
               .groupby(["campaign_id", "order_date"])
               .agg(orders=("order_id", "count"), revenue=("order_value_idr", "sum")))
first_party.index.names = ["campaign_id", "date"]
chk = d.set_index(["campaign_id", "date"]).join(first_party, how="left").fillna({"orders": 0, "revenue": 0})

conv_match = (chk["conversions"] == chk["orders"]).mean()
rev_match = (chk["revenue_idr"] == chk["revenue"]).mean()
print(f"campaign-days where platform conversions equal order count: {conv_match:.1%}")
print(f"campaign-days where platform revenue equals order revenue:  {rev_match:.1%}")

total_gap = chk["revenue"].sum() / chk["revenue_idr"].sum() - 1
print(f"total order revenue vs platform revenue: {total_gap:+.2%}")

# Referential integrity and basic rules
assert camp["campaign_id"].is_unique and cu["customer_id"].is_unique and o["order_id"].is_unique
assert d.duplicated(["date", "campaign_id"]).sum() == 0
assert (d["clicks"] <= d["impressions"]).all() and (d["conversions"] <= d["clicks"]).all()
assert (d["spend_idr"] >= 0).all() and (o["order_value_idr"] > 0).all()
assert o["customer_id"].dropna().isin(cu["customer_id"]).all()
assert d["date"].between(camp.set_index("campaign_id").loc[d["campaign_id"], "start_date"].values,
                         camp.set_index("campaign_id").loc[d["campaign_id"], "end_date"].values).all()
print("all integrity checks passed")

campaign-days where platform conversions equal order count: 100.0%
campaign-days where platform revenue equals order revenue:  99.8%
total order revenue vs platform revenue: -0.01%
all integrity checks passed

## Save clean tables and the cleaning log

In [12]:
camp_out = camp.copy()
d_out = d.drop(columns="orders_in_order_table").sort_values(["date", "campaign_id"])
cu_out = cu.sort_values("customer_id")
o_out = o.sort_values("order_id")

for name, df in [("campaigns", camp_out), ("daily_performance", d_out), ("customers", cu_out), ("orders", o_out)]:
    df.to_csv(CLEAN / f"{name}.csv", index=False, date_format="%Y-%m-%d")

summary = pd.DataFrame({
    "raw_rows": [len(raw[n]) for n in names],
    "clean_rows": [len(camp_out), len(d_out), len(cu_out), len(o_out)],
}, index=names)
summary["rows_removed"] = summary["raw_rows"] - summary["clean_rows"]
print(summary.to_string())

log_df = pd.DataFrame(cleaning_log)
lines = ["# Cleaning log", "",
         "Generated by `notebooks/01_data_cleaning.ipynb`. Every row is one problem found in `data/raw/` and what was done about it.", "",
         "| Table | Problem | Rows affected | Action |", "|---|---|---:|---|"]
for r in log_df.itertuples():
    lines.append(f"| {r.table} | {r.issue} | {r.rows_affected:,} | {r.action} |")
Path("../docs/cleaning_log.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"\n{len(log_df)} cleaning steps logged, {int(log_df['rows_affected'].sum()):,} row-level fixes in total")

                   raw_rows  clean_rows  rows_removed
campaigns                41          40             1
daily_performance      2481        2452            29
customers              7103        7033            70
orders                12908       12806           102

29 cleaning steps logged, 7,547 row-level fixes in total

## What this leaves for the analysis

- `dq_flags` marks every repaired row in `daily_performance` and `orders`. The analysis notebook will run its main numbers with and without flagged rows to show that the repairs do not drive the conclusions.
- Customers with `Unknown` city, gender or age group stay in the data and show up as their own group.
- Orders flagged `customer_unknown` count toward campaign revenue but are left out of customer-level metrics such as LTV.